# Stage 2 v3: 保守剪枝比例

**v2结果**: EKF 74.99% vs Random 74.75% (keep 58.75%) — 差距太小

**v3策略**: 降低剪枝比例, 保护accuracy
- `prune_quantile_min: 0.2 -> 0.05` (最激进只剪5%)
- `prune_quantile_max: 0.5 -> 0.2`  (最保守剪20%)

**预期**: keep 80-90%, EKF >> Random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ekf_adaptive_pruning')
print('[cwd]', os.getcwd())

## 只需要改 inference.py 里的一行超参

In [ ]:
inference_code = '''"""Inference with EKF-gated channel pruning (v3 - conservative)."""
import os
import sys
import argparse

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

from models.resnet18_cifar import resnet18_cifar
from models.ekf_pruner import EKFChannelGate, RandomChannelGate, softmax_entropy


def get_test_loader(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    test_set = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_tf)
    return DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


def wrap_layer_with_gate(layer_seq, gate_cls, gate_kwargs, device):
    new_modules = []
    gate_list = []
    for block in layer_seq:
        new_modules.append(block)
        out_ch = block.conv2.out_channels
        gate = gate_cls(num_channels=out_ch, device=device, **gate_kwargs).to(device)
        new_modules.append(gate)
        gate_list.append(gate)
    return nn.Sequential(*new_modules), gate_list


def install_gates(model, gate_cls, gate_kwargs, target_layers=("layer3", "layer4"), device="cuda"):
    all_gates = []
    for lname in target_layers:
        layer = getattr(model, lname)
        new_seq, gates = wrap_layer_with_gate(layer, gate_cls, gate_kwargs, device)
        setattr(model, lname, new_seq)
        all_gates.extend(gates)
    return all_gates


def evaluate_nogate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def evaluate_with_ekf_gates(model, gates, loader, device, entropy_norm=2.3):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    running_unc = 0.5
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            for g in gates:
                g.set_uncertainty(running_unc)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            ent = softmax_entropy(logits).mean().item()
            running_unc = min(1.0, ent / entropy_norm)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def evaluate_with_random_gates(model, gates, loader, device):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def load_base_model(ckpt_path, device):
    model = resnet18_cifar(num_classes=10).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


def run_comparison(ckpt_path, device="cuda"):
    loader = get_test_loader()

    print("\\n=== Baseline (no pruning) ===")
    m = load_base_model(ckpt_path, device)
    acc_base = evaluate_nogate(m, loader, device)
    print(f"acc = {acc_base:.2f}%")

    print("\\n=== EKF adaptive gating (conservative) ===")
    m = load_base_model(ckpt_path, device)
    # v3: conservative quantiles
    ekf_kwargs = dict(Q=1e-3, R=1e-2, alpha=3.0,
                      prune_quantile_min=0.05, prune_quantile_max=0.20,
                      warmup_steps=5)
    gates = install_gates(m, EKFChannelGate, ekf_kwargs,
                          target_layers=("layer3", "layer4"), device=device)
    acc_ekf, keep_ekf = evaluate_with_ekf_gates(m, gates, loader, device)
    avg_keep_ekf = sum(keep_ekf) / len(keep_ekf)
    print(f"acc = {acc_ekf:.2f}%,  avg_keep_ratio = {avg_keep_ekf:.3f}")
    print(f"per-gate keep: {[round(r, 3) for r in keep_ekf]}")

    match_keep = max(0.1, min(0.95, avg_keep_ekf))

    print("\\n=== Random gating (matched keep ratio) ===")
    m = load_base_model(ckpt_path, device)
    rand_kwargs = dict(keep_ratio=match_keep)
    gates = install_gates(m, RandomChannelGate, rand_kwargs,
                          target_layers=("layer3", "layer4"), device=device)
    acc_rand, keep_rand = evaluate_with_random_gates(m, gates, loader, device)
    avg_keep_rand = sum(keep_rand) / len(keep_rand)
    print(f"acc = {acc_rand:.2f}%,  avg_keep_ratio = {avg_keep_rand:.3f}")

    print("\\n=== Summary ===")
    print(f"  No pruning:      {acc_base:.2f}%")
    print(f"  EKF gating:      {acc_ekf:.2f}%  (keep {avg_keep_ekf:.2%})")
    print(f"  Random (match):  {acc_rand:.2f}%  (keep {avg_keep_rand:.2%})")
    print(f"  EKF advantage:   {acc_ekf - acc_rand:+.2f}% over random")
    print(f"  Accuracy drop:   {acc_base - acc_ekf:.2f}%")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ckpt", type=str, default="./checkpoints/resnet18_cifar_base.pth")
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")
    run_comparison(args.ckpt, device=device)
'''

with open('src/inference.py', 'w') as f:
    f.write(inference_code)
print('inference.py v3 写入完成')

In [ ]:
!python src/inference.py

## 这次的预期

```
No pruning:      94.97%
EKF gating:      90-93%  (keep 80-90%)
Random (match):  比EKF低 2-4%
EKF advantage:   +2% 以上
```

如果EKF advantage > +2%, 说明机制work, 可以继续深挖.
如果还是只比random高0.5%以下, 就要走选项B (换BN gamma做proxy).